In [1]:
from data.datafinder import DataFinder

import pandas as pd
from functools import partial
import json
import random
from dataclasses import asdict
from pathlib import Path

from dataset_recsys.ingestion.fetch_gems_datasets import DatasetProfile
from dataset_recsys.embeddings import build_embedding_text, build_raw_embedding_text, encode_texts
from dataset_recsys.retrieval import build_recommendations
from dataset_recsys.utils.bedrock import enrich_batch
from dataset_recsys.utils.text_preprocessing import preprocess_profiles_field
from dataset_recsys.workflows.full_batch_rebuild import FullBatchRebuildWorkflow

from recs_metrics.item_item import recall_at_n, tndcg_at_n

EVAL_CUTOFFS = [10, 20, 50]
CONNECTED_QUERY_COUNT = 50
ADDITIONAL_DATASET_COUNT = 100
RANDOM_SEED = 42
rng = random.Random(RANDOM_SEED)

DEMO_OUTPUT_DIR = Path("data/datafinder/demo_outputs")
DEMO_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ENRICHMENT_LLM = "claude-sonnet-4-6"
PROMPT_VERSION = "catalog_summary_v1"
EMBEDDING_MODEL = "allenai/specter2_base"

In [2]:
df = DataFinder()
data = df.get()
corpus = data["corpus"]
ground_truth_links = df.get_links_from_queries()

##### We select a fixed number of query datasets, add all their linked neighbors, and then add a small number of extra distractor datasets to make the problem more realistic without running Bedrock on the full corpus.

In [ ]:
all_query_ids = sorted(ground_truth_links.keys())
selected_query_ids = rng.sample(all_query_ids, min(CONNECTED_QUERY_COUNT, len(all_query_ids)))
selected_dataset_ids = set(selected_query_ids)
for query_id in selected_query_ids:
    selected_dataset_ids.update(ground_truth_links.get(query_id, []))

remaining_dataset_ids = sorted(set(corpus["id"]) - selected_dataset_ids)
extra_ids = rng.sample(remaining_dataset_ids, min(ADDITIONAL_DATASET_COUNT, len(remaining_dataset_ids)))
selected_dataset_ids.update(extra_ids)

corpus = corpus[corpus["id"].isin(selected_dataset_ids)].copy()

selected_query_id_set = set(selected_query_ids)
ground_truth_links = {
    query_id: {target_id for target_id in targets if target_id in selected_dataset_ids}
    for query_id, targets in ground_truth_links.items()
    if query_id in selected_query_id_set
}

##### We transform heterogeneous Datafinder metadata into the unified DatasetProfile format used by the recommender pipeline.

In [4]:
def datafinder_to_dataset_profiles(row: pd.Series) -> DatasetProfile:
    title = row["title"] if pd.notna(row["title"]) else str(row["id"])
    description = row["description"] if pd.notna(row["description"]) else ""

    keyword: list[str] = []
    if isinstance(row["tasks"], list):
        keyword.extend(str(task) for task in row["tasks"] if task)

    return DatasetProfile(
        id=str(row["id"]),
        title=str(title),
        headline="",
        description=str(description),
        keywords=", ".join(keyword),
        field_of_science="",
        catalog_summary="", # to generate with LLM
    )

profiles = corpus.apply(datafinder_to_dataset_profiles, axis=1).tolist()
profiles = preprocess_profiles_field(profiles, field_name="description")

In [5]:
print(f"Total datasets sampled: {len(profiles)}")

Total datasets sampled: 397


##### Run the full batch workflow (demo mode)
##### We reuse the same workflow orchestration as the portal recommender, but without storing results DBs.

In [6]:
workflow_outputs: dict[str, object] = {
    "dataset_profiles": None,
    "recommendations": None,
}


def fetch_catalog_step() -> list[DatasetProfile]:
    return profiles

def preprocess_catalog_step(catalog: list[DatasetProfile]) -> list[DatasetProfile]:
    processed_catalog = preprocess_profiles_field(catalog, field_name="catalog_summary")
    workflow_outputs["dataset_profiles"] = processed_catalog
    return processed_catalog

def generate_embeddings_step(catalog: list[DatasetProfile]):
    embedding_texts = [build_embedding_text(profile) for profile in catalog]
    return encode_texts(embedding_texts, model_name=EMBEDDING_MODEL)

def compute_recommendations_step(embeddings, catalog: list[DatasetProfile]):
    return build_recommendations(catalog, embeddings, top_k=None)

def write_recommendations_step(recommendations):
    workflow_outputs["recommendations"] = recommendations

workflow = FullBatchRebuildWorkflow(
    fetch_catalog=fetch_catalog_step,
    enrich_catalog=partial(
        enrich_batch,
        llm=ENRICHMENT_LLM,
        prompt_version=PROMPT_VERSION,
    ),
    preprocess_catalog=preprocess_catalog_step,
    generate_embeddings=generate_embeddings_step,
    compute_recommendations=compute_recommendations_step,
    write_recommendations=write_recommendations_step,
)

artifacts = workflow.run()

print("\n[DEMO] Workflow artifacts summary:")

print(f"- Started at: {artifacts.started_at}")
print(f"- Finished at: {artifacts.finished_at}")
print(f"- Duration: {artifacts.duration_seconds:.2f} seconds")
print(f"- Raw datasets fetched: {artifacts.raw_catalog_size}")
print(f"- Processed datasets: {artifacts.processed_catalog_size}")
print(f"- Recommendation lists produced: {artifacts.recommendation_count}")

recommendations, dataset_profiles_processed = workflow_outputs["recommendations"], workflow_outputs["dataset_profiles"] 

with open(DEMO_OUTPUT_DIR / "datafinder_profiles.json", "w", encoding="utf-8") as f:
    json.dump([asdict(profile) for profile in dataset_profiles_processed], f, ensure_ascii=False, indent=2)


with open(DEMO_OUTPUT_DIR / "datafinder_recommendations.json", "w", encoding="utf-8") as f:
    json.dump(recommendations, f, ensure_ascii=False, indent=2)


Enriching: Parameterisation of a stochastic model for human face identification
Invoking Bedrock model/profile: eu.anthropic.claude-sonnet-4-6 in region eu-central-1
Enriching: A Database for Handwritten Text Recognition Research
Invoking Bedrock model/profile: eu.anthropic.claude-sonnet-4-6 in region eu-central-1
Enriching: Automating the Construction of Internet Portals with Machine Learning
Invoking Bedrock model/profile: eu.anthropic.claude-sonnet-4-6 in region eu-central-1
Enriching: Locating blood vessels in retinal images by piecewise threshold probing of a matched filter response
Invoking Bedrock model/profile: eu.anthropic.claude-sonnet-4-6 in region eu-central-1
Enriching: CBSD68
Invoking Bedrock model/profile: eu.anthropic.claude-sonnet-4-6 in region eu-central-1
Enriching: A Database of Human Segmented Natural Images and its Application to Evaluating Segmentation Algorithms and Measuring Ecological Statistics
Invoking Bedrock model/profile: eu.anthropic.claude-sonnet-4-6 in

##### Run the full batch workflow without LLM enrichment
##### We use the same workflow orchestration, but skip Bedrock and build embeddings directly from the raw metadata fields.

In [7]:
baseline_workflow_outputs: dict[str, object] = {
    "recommendations": None,
}


def skip_enrichment_step(catalog: list[DatasetProfile]) -> list[DatasetProfile]:
    return catalog

def baseline_preprocess_catalog_step(catalog: list[DatasetProfile]) -> list[DatasetProfile]:
    return catalog

def baseline_generate_embeddings_step(catalog: list[DatasetProfile]):
    embedding_texts = [build_raw_embedding_text(profile) for profile in catalog]
    return encode_texts(embedding_texts, model_name=EMBEDDING_MODEL)

def baseline_write_recommendations_step(recommendations):
    baseline_workflow_outputs["recommendations"] = recommendations

baseline_workflow = FullBatchRebuildWorkflow(
    fetch_catalog=fetch_catalog_step,
    enrich_catalog=skip_enrichment_step,
    preprocess_catalog=baseline_preprocess_catalog_step,
    generate_embeddings=baseline_generate_embeddings_step,
    compute_recommendations=compute_recommendations_step,
    write_recommendations=baseline_write_recommendations_step,
)

baseline_artifacts = baseline_workflow.run()
baseline_recommendations = baseline_workflow_outputs["recommendations"]

print("\n[DEMO] Baseline workflow artifacts summary:")

print(f"- Started at: {baseline_artifacts.started_at}")
print(f"- Finished at: {baseline_artifacts.finished_at}")
print(f"- Duration: {baseline_artifacts.duration_seconds:.2f} seconds")
print(f"- Raw datasets fetched: {baseline_artifacts.raw_catalog_size}")
print(f"- Processed datasets: {baseline_artifacts.processed_catalog_size}")
print(f"- Recommendation lists produced: {baseline_artifacts.recommendation_count}")

with open(DEMO_OUTPUT_DIR / "datafinder_recommendations_no_llm.json", "w", encoding="utf-8") as f:
    json.dump(baseline_recommendations, f, ensure_ascii=False, indent=2)


[DEMO] Baseline workflow artifacts summary:
- Started at: 2026-04-01 23:22:33.949422
- Finished at: 2026-04-01 23:25:28.835788
- Duration: 174.89 seconds
- Raw datasets fetched: 397
- Processed datasets: 397
- Recommendation lists produced: 397


##### We evaluate the generated ranking lists against the Datafinder ground-truth links.

In [9]:
with open(DEMO_OUTPUT_DIR / "datafinder_recommendations.json", "r", encoding="utf-8") as f:
    recommendations = json.load(f)

with open(DEMO_OUTPUT_DIR / "datafinder_recommendations_no_llm.json", "r", encoding="utf-8") as f:
    baseline_recommendations = json.load(f)

In [ ]:
results_llm = {}
for n in EVAL_CUTOFFS:
    predictions = {
        item["id"]: [rec["id"] for rec in item.get("recommendations", [])][:n]
        for item in recommendations
    }
    results_llm[f"Recall@{n}"] = recall_at_n(predictions, ground_truth_links, n=n)
    results_llm[f"TNDCG@{n}"] = tndcg_at_n(predictions, ground_truth_links, n=n)

results_no_llm = {}
for n in EVAL_CUTOFFS:
    predictions = {
        item["id"]: [rec["id"] for rec in item.get("recommendations", [])][:n]
        for item in baseline_recommendations
    }
    results_no_llm[f"Recall@{n}"] = recall_at_n(predictions, ground_truth_links, n=n)
    results_no_llm[f"TNDCG@{n}"] = tndcg_at_n(predictions, ground_truth_links, n=n)

results_df = pd.DataFrame(
    [results_llm, results_no_llm],
    index=["Claude + SPECTER", "SPECTER"],
)
results_df

,Recall@10,TNDCG@10,Recall@20,TNDCG@20,Recall@50,TNDCG@50
Claude + SPECTER,0.541215,0.759029,0.671412,0.737650,0.812937,0.711068
SPECTER,0.476519,0.764360,0.635722,0.743442,0.806670,0.713717
